# SituatiONION V4: Structural Controls and Large-Scale Replication

**Question:** does held-out situation separation survive strict surface controls on a 300-triple dataset? **H1** tests whether layers 23-26 exceed adjacent layers; **H2** tests for earlier or broader reliable emergence.

The model-derived conclusion is generated only after all prespecified readouts and held-out tests run.

In [ ]:
from pathlib import Path
import hashlib, re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

ROOT = Path.cwd(); DATA = ROOT / 'data' / 'matched_triples.csv'; OUT = ROOT / 'results' / 'layer_curves'
OUT.mkdir(parents=True, exist_ok=True)
SEED, N_BOOT = 7, 2000
H1_CORE = tuple(range(23, 27)); H1_NEIGHBORS = tuple(range(19, 23)) + tuple(range(27, 31))
rng = np.random.default_rng(SEED)
if not DATA.exists(): raise FileNotFoundError('Run `python3 structural_controls.py` first.')
triples = pd.read_csv(DATA); dataset_sha256 = hashlib.sha256(DATA.read_bytes()).hexdigest()
gap_columns = [c for c in triples if c.startswith('absolute_') and c.endswith('_gap')]
assert 200 <= len(triples) <= 500
assert {'dev', 'test_template', 'test_entity', 'test_both'} <= set(triples.split)
print(f'Frozen dataset: {len(triples)} triples | SHA-256: {dataset_sha256}')
display(pd.crosstab(triples.manipulation, triples.split))
display(triples.groupby('manipulation')[gap_columns].mean().round(3))
display(triples[['identifier','manipulation','split','base','paraphrase','counterfactual']].sample(8, random_state=SEED))


## 1. Data and Matching Protocol

`structural_controls.py` creates five manipulation types, three template families, and independent held-out template/entity splits. The table above validates unigram, bigram, length, token-edit, and structural-edit matching before any model result is inspected. `data/matched_triples.csv` is frozen input for this notebook.

In [ ]:
MODEL_NAME = 'gpt2-xl'; CACHE = OUT / 'gpt2xl_hidden_states.pt'
RUN_MODEL = False  # Set True only on a machine with at least 7 GB free for model weights.

def extract_runs(frame):
    device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
    dtype = torch.float16 if device == 'cuda' else torch.float32
    tok = GPT2Tokenizer.from_pretrained(MODEL_NAME); tok.pad_token = tok.eos_token
    model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device).eval(); runs = {}
    with torch.no_grad():
        for row in frame.itertuples(index=False):
            runs[row.identifier] = {}
            for variant in ('base','paraphrase','counterfactual'):
                text = getattr(row, variant); output = model(**tok(text, return_tensors='pt').to(device), output_hidden_states=True, use_cache=False)
                runs[row.identifier][variant] = {'text': text, 'hidden_states': tuple(x[0].detach().float().cpu() for x in output.hidden_states)}
    return tok, runs

if CACHE.exists():
    saved = torch.load(CACHE, map_location='cpu', weights_only=False); tokenizer, RUNS = saved['tokenizer'], saved['runs']
elif RUN_MODEL:
    tokenizer, RUNS = extract_runs(triples); torch.save({'tokenizer': tokenizer, 'runs': RUNS, 'dataset_sha256': dataset_sha256}, CACHE)
else:
    RUNS = None; print('Set RUN_MODEL = True to create the hidden-state cache.')


## 2. Prespecified Readouts

At every layer, V4 compares mean pooling, max pooling, final token, changed-field token, event token, and an ordered agent/recipient readout. Positive values always mean `cos(base, paraphrase) - cos(base, counterfactual) > 0`.

In [ ]:
READOUTS = ('mean_pool','max_pool','final_token','changed_token','event_token','agent_recipient')
LABELS = {'mean_pool':'Mean pool','max_pool':'Max pool','final_token':'Final token','changed_token':'Changed token','event_token':'Event token','agent_recipient':'Agent/recipient'}
EVENT = {'agent_recipient':('gave','gave','gave'),'cause':('called','called','called'),'temporal':('cleaned','cleaned','cleaned'),'polarity':('repair','mend','fix'),'event_state':('carried','transported','dropped')}
CHANGED = {'temporal':('after','once','before'),'polarity':('repair','mend','not'),'event_state':('carried','transported','dropped')}

def pos(text, surface):
    hits = list(re.finditer(rf'(?<!\w){re.escape(surface)}(?!\w)', text, re.I))
    if not hits: raise ValueError(f'{surface!r} not found in {text!r}')
    return len(tokenizer.encode(text[:hits[-1].end()], add_special_tokens=False)) - 1
def surf(row, variant, kind):
    i = ('base','paraphrase','counterfactual').index(variant)
    if kind == 'event': return EVENT[row.manipulation][i]
    if row.manipulation == 'agent_recipient': return row.recipient if variant == 'counterfactual' else row.agent
    if row.manipulation == 'cause': return row.agent if variant == 'counterfactual' else row.recipient
    return CHANGED[row.manipulation][i]
def vector(row, variant, layer, readout):
    run = RUNS[row.identifier][variant]; h = run['hidden_states'][layer + 1]
    if readout == 'mean_pool': v = h.mean(0)
    elif readout == 'max_pool': v = h.max(0).values
    elif readout == 'final_token': v = h[-1]
    elif readout == 'changed_token': v = h[pos(run['text'], surf(row, variant, 'changed'))]
    elif readout == 'event_token': v = h[pos(run['text'], surf(row, variant, 'event'))]
    else:
        agent, recipient = (row.recipient, row.agent) if row.manipulation == 'agent_recipient' and variant == 'counterfactual' else (row.agent, row.recipient)
        v = torch.cat((h[pos(run['text'], agent)], h[pos(run['text'], recipient)]))
    return v.numpy()
def cosine(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))
def collect_scores(frame):
    if RUNS is None: raise RuntimeError('Create or load the hidden-state cache first.')
    rows=[]; n_layers=len(next(iter(RUNS.values()))['base']['hidden_states'])-1
    for row in frame.itertuples(index=False):
        for readout in READOUTS:
            for layer in range(n_layers):
                b=vector(row,'base',layer,readout); p=cosine(b,vector(row,'paraphrase',layer,readout)); c=cosine(b,vector(row,'counterfactual',layer,readout))
                rows.append({'identifier':row.identifier,'split':row.split,'manipulation':row.manipulation,'readout':readout,'layer':layer,'paraphrase_similarity':p,'counterfactual_similarity':c,'separation':p-c})
    return pd.DataFrame(rows)
pair_scores = collect_scores(triples); pair_scores.to_csv(OUT/'pair_scores.csv', index=False)


In [ ]:
def bootstrap_curves(frame, groups):
    rows=[]
    for keys, group in frame.groupby(groups+['layer'], sort=False):
        keys = keys if isinstance(keys, tuple) else (keys,); values=group.separation.to_numpy()
        draws=values[rng.integers(0,len(values),size=(N_BOOT,len(values)))].mean(1)
        rows.append(dict(zip(groups+['layer'],keys), n=len(values), mean=values.mean(), ci_low=np.quantile(draws,.025), ci_high=np.quantile(draws,.975), effect_size=values.mean()/(values.std(ddof=1)+1e-12)))
    return pd.DataFrame(rows)
heldout = pair_scores.query("split != 'dev'").copy()
curves = bootstrap_curves(heldout,['readout']); by_manipulation = bootstrap_curves(heldout,['readout','manipulation'])
curves.to_csv(OUT/'heldout_layer_curves.csv',index=False); by_manipulation.to_csv(OUT/'heldout_manipulation_curves.csv',index=False)

fig, axes=plt.subplots(2,3,figsize=(15,8),sharex=True,sharey=True)
for ax, readout in zip(axes.flat,READOUTS):
    x=curves.query('readout == @readout'); ax.plot(x.layer,x['mean']); ax.fill_between(x.layer,x.ci_low,x.ci_high,alpha=.2); ax.axhline(0,color='black',lw=.8); ax.axvspan(23,26,color='orange',alpha=.12); ax.set(title=LABELS[readout],xlabel='GPT-2 XL block',ylabel='Held-out separation')
fig.suptitle('V4: all prespecified readouts',y=1.02); fig.tight_layout(); fig.savefig(OUT/'heldout_readout_curves.png',dpi=180,bbox_inches='tight'); plt.show()

fig, axes=plt.subplots(3,2,figsize=(14,10),sharex=True,sharey=True)
for ax, readout in zip(axes.flat,READOUTS):
    for manipulation,x in by_manipulation.query('readout == @readout').groupby('manipulation'): ax.plot(x.layer,x['mean'],label=manipulation)
    ax.axhline(0,color='black',lw=.8); ax.axvspan(23,26,color='orange',alpha=.12); ax.set(title=LABELS[readout],xlabel='GPT-2 XL block',ylabel='Held-out separation')
axes[0,0].legend(fontsize=8); fig.suptitle('V4: manipulation-specific curves',y=1.01); fig.tight_layout(); fig.savefig(OUT/'heldout_manipulation_curves.png',dpi=180,bbox_inches='tight'); plt.show()


## 3. H1, H2, and Conclusion

H1 is supported only if the held-out 23-26 average exceeds the adjacent control layers with a positive 95% bootstrap interval. H2 records the first and last reliably positive held-out layers for every readout. The conclusion below concerns geometry, not causal use.

In [ ]:
def boot_mean(values):
    draws=values[rng.integers(0,len(values),size=(N_BOOT,len(values)))].mean(1); return values.mean(),np.quantile(draws,.025),np.quantile(draws,.975)
h1=[]
for readout,g in heldout.groupby('readout'):
    matrix=g.pivot(index='identifier',columns='layer',values='separation'); contrast=(matrix.loc[:,H1_CORE].mean(1)-matrix.loc[:,H1_NEIGHBORS].mean(1)).to_numpy(); mean,low,high=boot_mean(contrast); h1.append({'readout':readout,'core_minus_neighbors':mean,'ci_low':low,'ci_high':high,'supports_h1':low>0})
h1=pd.DataFrame(h1)
h2=[]
for readout,g in curves.groupby('readout'):
    positive=g.query('ci_low > 0').layer.to_list(); h2.append({'readout':readout,'first_reliably_positive_layer':min(positive) if positive else np.nan,'last_reliably_positive_layer':max(positive) if positive else np.nan,'positive_layer_count':len(positive)})
h2=pd.DataFrame(h2); h1.to_csv(OUT/'h1_results.csv',index=False); h2.to_csv(OUT/'h2_results.csv',index=False)
display(h1.assign(readout=h1.readout.map(LABELS)).round(4)); display(h2.assign(readout=h2.readout.map(LABELS)))
supported=h1[h1.supports_h1].readout.map(LABELS).to_list(); detected=h2.dropna(subset=['first_reliably_positive_layer'])
emergence='No prespecified readout showed reliable held-out separation.' if detected.empty else 'Reliable held-out separation: ' + '; '.join(f"{LABELS[r.readout]} at layers {int(r.first_reliably_positive_layer)}-{int(r.last_reliably_positive_layer)}" for r in detected.itertuples()) + '.'
print('V4 conclusion')
print(emergence)
print('H1 supported for: ' + ', '.join(supported) + '.' if supported else 'H1 was not supported by any prespecified readout.')
print('This is evidence about representational geometry under strict controls, not causal use.')
